![Universidad Espíritu Santo](https://raw.githubusercontent.com/andresrubiop/miar0525-estudiantes/main/utils/logo-uees-color.png)

<div style="background:#821436;color:#FFFFFF;padding:14px 18px;border-radius:10px;margin:6px 0 12px 0"><div style="font-size:12px;letter-spacing:.08em;text-transform:uppercase;opacity:.9">Aprendizaje Automático · MIAR0525 · Semana 2 · Notebook del estudiante · no calificable</div><div style="font-size:22px;font-weight:700;margin-top:4px">E2.2 · Regresión logística y calibración</div><div style="font-size:12px;opacity:.9;margin-top:4px">Postgrado · Maestría en Inteligencia Artificial · UEES</div></div>

| | |
|---|---|
| **Objetivo** | Programar la regresión logística, interpretar sus probabilidades y comprobar si están calibradas antes de usarlas para decidir. |
| **Resultado de aprendizaje** | RDA1 · competencias CG-G1 y CE-G1 |
| **Duración** | ≈ 4 h |
| **Teoría** | Manual M2 §3–4 · Animaciones A2.3 y A2.4 · Video V2.1 |
| **Datos** | Sintéticos · **Bank Marketing** (OpenML 1461: 45 211 llamadas de una campaña bancaria, 16 variables, 11.7 % contrata el depósito) |

**Niveles:** 1 · sigmoide, log-loss y gradiente desde cero → 2 · scikit-learn, `C` y `class_weight` → 3 · Bank Marketing: PR-AUC, calibración y decisión por valor esperado → 4 · reto: Platt frente a isotónica.

## 0 · Configuración

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import sklearn
from cycler import cycler

warnings.filterwarnings("ignore", category=FutureWarning)
SEED = 2026
UEES = {"vino": "#821436", "azul": "#1F6F8B", "ocre": "#C28E0E", "verde": "#3A7D44", "gris": "#77787B"}
plt.rcParams.update({
    "axes.prop_cycle": cycler(color=list(UEES.values())), "axes.titlecolor": UEES["vino"],
    "axes.titleweight": "bold", "axes.edgecolor": UEES["gris"], "axes.grid": True, "grid.color": "#EEE8EA",
    "axes.spines.top": False, "axes.spines.right": False, "figure.dpi": 110, "legend.frameon": False,
})
print(f"scikit-learn {sklearn.__version__} · semilla {SEED}")

## Nivel 1 · Desde cero (prelaboratorio)

Dos clases en el plano, solapadas. El modelo es $p(x) = \sigma(w^\top x + b)$ con $\sigma(z) = 1/(1+e^{-z})$, y se entrena minimizando la **log-loss**:

$$L(w, b) = -\frac1n\sum_i \big[y_i\log p_i + (1-y_i)\log(1-p_i)\big], \qquad \nabla_w L = \frac1n X^\top(p - y), \quad \frac{\partial L}{\partial b} = \frac1n\sum_i (p_i - y_i)$$

In [ ]:
from sklearn.datasets import make_classification

X2, y2 = make_classification(n_samples=400, n_features=2, n_redundant=0, n_informative=2, n_clusters_per_class=1,
                             class_sep=1.0, flip_y=0.03, random_state=SEED)


def sigmoide(z):
    # TODO: implementa la sigmoide (usa np.exp).
    return ...


def log_loss_propia(y, p, eps=1e-12):
    # TODO: implementa la log-loss promedio (recorta p a [eps, 1 - eps] para evitar log(0)).
    p = ...
    return ...


def entrenar_logistica(X, y, eta=0.5, iters=3000):
    w, b, hist = np.zeros(X.shape[1]), 0.0, []
    for _ in range(iters):
        # TODO: calcula p, el gradiente de w y de b, y actualiza; guarda la log-loss.
        p = ...
        w = ...
        b = ...
        hist.append(...)
    return w, b, np.array(hist)


w_p, b_p, hist_p = entrenar_logistica(X2, y2)
print(f"w = {np.round(w_p, 4)} · b = {b_p:.4f} · log-loss final {hist_p[-1]:.4f}")

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(10.5, 4))
xx, yy = np.meshgrid(np.linspace(X2[:, 0].min() - 0.5, X2[:, 0].max() + 0.5, 200), np.linspace(X2[:, 1].min() - 0.5, X2[:, 1].max() + 0.5, 200))
pp = sigmoide(np.c_[xx.ravel(), yy.ravel()] @ w_p + b_p).reshape(xx.shape)
cf = axes[0].contourf(xx, yy, pp, levels=np.linspace(0, 1, 11), cmap="RdBu_r", alpha=0.35)
axes[0].contour(xx, yy, pp, levels=[0.5], colors="#1C1A1B", linewidths=2)
axes[0].scatter(X2[:, 0], X2[:, 1], c=np.where(y2 == 1, UEES["vino"], UEES["azul"]), s=12)
axes[0].set(title="Probabilidad y frontera p = 0.5", xlabel="x₁", ylabel="x₂")
fig.colorbar(cf, ax=axes[0], label="p(y = 1)")
axes[1].plot(hist_p, color=UEES["ocre"])
axes[1].set(title="Log-loss por iteración", xlabel="iteración", ylabel="log-loss", xscale="log")
plt.tight_layout()
plt.show()

## Nivel 2 · Con scikit-learn
`LogisticRegression` regulariza por defecto con `C = 1` ($C = 1/\lambda$). Para comparar con tu versión sin penalización usamos `C` muy grande.

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import log_loss

lr_sk = LogisticRegression(C=1e6, max_iter=5000).fit(X2, y2)
print(f"scikit-learn: w = {np.round(lr_sk.coef_[0], 4)} · b = {lr_sk.intercept_[0]:.4f}")
assert np.allclose(lr_sk.coef_[0], w_p, atol=0.02) and np.isclose(lr_sk.intercept_[0], b_p, atol=0.02), "Tu gradiente no llegó al mismo óptimo."
assert np.isclose(log_loss(y2, lr_sk.predict_proba(X2)[:, 1]), log_loss_propia(y2, lr_sk.predict_proba(X2)[:, 1])), "Tu log-loss no coincide."
print("✓ Tu regresión logística coincide con scikit-learn")

### 2.1 El efecto de `C`
Con menos `C` (más regularización) los coeficientes se encogen y las probabilidades se acercan a 0.5 (animación A2.3).

In [ ]:
filas = []
for C in (0.001, 0.01, 0.1, 1, 100):
    m = LogisticRegression(C=C, max_iter=5000).fit(X2, y2)
    p = m.predict_proba(X2)[:, 1]
    filas.append({"C": C, "‖w‖": round(float(np.linalg.norm(m.coef_)), 3), "p mín": round(float(p.min()), 3), "p máx": round(float(p.max()), 3),
                  "log-loss": round(log_loss(y2, p), 4), "exactitud": round(m.score(X2, y2), 3)})
pd.DataFrame(filas)

### 2.2 `class_weight="balanced"` mueve las probabilidades
Con clases desbalanceadas, pesar más la clase minoritaria sube el recall a umbral 0.5, **pero las probabilidades dejan de significar lo que dicen**. Lo vemos con el diagrama de fiabilidad.

In [ ]:
from sklearn.calibration import CalibrationDisplay
from sklearn.model_selection import train_test_split

X_d, y_d = make_classification(n_samples=6000, n_features=6, n_informative=4, weights=[0.9, 0.1], class_sep=0.8, random_state=SEED)
Xd_tr, Xd_te, yd_tr, yd_te = train_test_split(X_d, y_d, test_size=0.4, stratify=y_d, random_state=SEED)
normal = LogisticRegression(max_iter=2000).fit(Xd_tr, yd_tr)
balanceado = LogisticRegression(max_iter=2000, class_weight="balanced").fit(Xd_tr, yd_tr)

fig, ax = plt.subplots(figsize=(5.2, 4.6))
CalibrationDisplay.from_estimator(normal, Xd_te, yd_te, n_bins=10, name="sin pesos", ax=ax)
CalibrationDisplay.from_estimator(balanceado, Xd_te, yd_te, n_bins=10, name="class_weight='balanced'", ax=ax)
ax.set(title="Diagrama de fiabilidad (prueba)")
plt.show()
p_n, p_b = normal.predict_proba(Xd_te)[:, 1], balanceado.predict_proba(Xd_te)[:, 1]
print(f"prevalencia real {yd_te.mean():.3f} · probabilidad media predicha: sin pesos {p_n.mean():.3f} · balanceado {p_b.mean():.3f}")

**Qué observar.** El modelo balanceado queda **por debajo de la diagonal**: sobreestima la probabilidad de la clase minoritaria en todo el rango (en promedio predice mucho más que la prevalencia real). Sirve para ordenar y para decidir con un umbral ajustado, no para leer sus probabilidades.

## Nivel 3 · Datos reales: Bank Marketing

Un banco portugués llamó por teléfono a sus clientes para ofrecer un depósito a plazo. OpenML entrega las columnas como `V1`–`V16`; les devolvemos sus nombres originales (UCI).

**Fuga:** `duration` (duración de la llamada) solo se conoce **después** de llamar y casi determina el resultado. Los autores del dataset piden excluirla para un modelo realista. La quitamos antes de todo, como en la semana 1.

In [ ]:
from sklearn.datasets import fetch_openml

banco = fetch_openml(data_id=1461, as_frame=True)
nombres = ["age", "job", "marital", "education", "default", "balance", "housing", "loan", "contact", "day", "month",
           "duration", "campaign", "pdays", "previous", "poutcome"]
Xb = banco.data.set_axis(nombres, axis=1).drop(columns=["duration"])
yb = (banco.target == "2").astype(int)
print(f"{len(Xb)} llamadas · {Xb.shape[1]} variables · {yb.mean():.1%} contratan el depósito")
Xb.head()

In [ ]:
from sklearn.compose import ColumnTransformer, make_column_selector
from sklearn.metrics import average_precision_score, brier_score_loss, roc_auc_score
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

Xb_tr, Xb_te, yb_tr, yb_te = train_test_split(Xb, yb, test_size=0.25, stratify=yb, random_state=SEED)
prep = ColumnTransformer([
    ("num", StandardScaler(), make_column_selector(dtype_include="number")),
    ("cat", OneHotEncoder(handle_unknown="ignore"), make_column_selector(dtype_include="category")),
])
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=SEED)
modelos = {"logística": make_pipeline(prep, LogisticRegression(max_iter=3000)),
           "logística balanceada": make_pipeline(prep, LogisticRegression(max_iter=3000, class_weight="balanced"))}
p_oof = {k: cross_val_predict(m, Xb_tr, yb_tr, cv=cv, method="predict_proba")[:, 1] for k, m in modelos.items()}
tabla = pd.DataFrame({k: {"ROC-AUC": roc_auc_score(yb_tr, p), "PR-AUC": average_precision_score(yb_tr, p),
                          "Brier": brier_score_loss(yb_tr, p), "p media": p.mean()} for k, p in p_oof.items()}).T.round(4)
print(f"prevalencia = {yb_tr.mean():.4f} (PR-AUC del azar)")
tabla

Los dos modelos **ordenan casi igual** (AUC muy parecidas), pero el balanceado tiene peor Brier y una probabilidad media muy superior a la prevalencia.

### 3.1 Decidir con probabilidades: valor esperado
Supongamos que cada llamada cuesta 1 unidad y que un depósito contratado deja 8. Llamar conviene si $8\,p - 1 > 0$, es decir si $p > 1/8 = 0.125$. **Esta regla solo funciona si $p$ está calibrada.**

In [ ]:
GANANCIA, COSTO = 8, 1
t_ev = COSTO / GANANCIA


def beneficio(y, p, umbral):
    llamar = p >= umbral
    return int(GANANCIA * np.sum(y[llamar]) - COSTO * np.sum(llamar)), int(np.sum(llamar))


for k, m in modelos.items():
    m.fit(Xb_tr, yb_tr)
p_te = {k: m.predict_proba(Xb_te)[:, 1] for k, m in modelos.items()}
filas = []
for k, p in p_te.items():
    for t in (0.5, t_ev):
        b, n_llam = beneficio(yb_te.to_numpy(), p, t)
        filas.append({"modelo": k, "umbral": t, "llamadas": n_llam, "beneficio": b})
umbrales = np.linspace(0.02, 0.9, 89)
mejor = {k: max(umbrales, key=lambda t: beneficio(yb_tr.to_numpy(), p_oof[k], t)[0]) for k in modelos}
for k in modelos:
    b, n_llam = beneficio(yb_te.to_numpy(), p_te[k], mejor[k])
    filas.append({"modelo": k, "umbral": round(float(mejor[k]), 3), "llamadas": n_llam, "beneficio": b})
tabla_benef = pd.DataFrame(filas)
tabla_benef

**Qué observar.** Con la logística sin pesos, el umbral teórico 0.125 queda muy cerca del mejor umbral encontrado por validación cruzada: sus probabilidades son confiables. Con el modelo balanceado, 0.125 hace llamar a casi todos y el beneficio cae; hay que buscar su umbral a ciegas. **Calibrar es lo que permite pasar de la probabilidad a la decisión con una fórmula.**

In [ ]:
fig, ax = plt.subplots(figsize=(5.2, 4.6))
for k, p in p_te.items():
    CalibrationDisplay.from_predictions(yb_te, p, n_bins=10, strategy="quantile", name=k, ax=ax)
ax.set(title="Fiabilidad en prueba · Bank Marketing")
plt.show()

## Nivel 4 · Reto: calibrar el modelo balanceado
`CalibratedClassifierCV` aprende una transformación monótona sobre folds que el modelo no vio: **Platt** (`method="sigmoid"`) o **isotónica**. El orden no cambia (la AUC casi no se mueve), pero el Brier y la fiabilidad mejoran.

In [ ]:
from sklearn.calibration import CalibratedClassifierCV

filas = []
cal = {}
for metodo in ("sigmoid", "isotonic"):
    cal[metodo] = CalibratedClassifierCV(modelos["logística balanceada"], method=metodo, cv=5).fit(Xb_tr, yb_tr)
for nombre, p in [("balanceada sin calibrar", p_te["logística balanceada"]),
                  ("balanceada + Platt", cal["sigmoid"].predict_proba(Xb_te)[:, 1]),
                  ("balanceada + isotónica", cal["isotonic"].predict_proba(Xb_te)[:, 1]),
                  ("logística sin pesos", p_te["logística"])]:
    filas.append({"modelo": nombre, "ROC-AUC": roc_auc_score(yb_te, p), "Brier": brier_score_loss(yb_te, p),
                  "p media": p.mean(), "beneficio con 0.125": beneficio(yb_te.to_numpy(), p, t_ev)[0]})
tabla_cal = pd.DataFrame(filas).set_index("modelo").round(4)
tabla_cal

**Qué observar.** Después de calibrar, la probabilidad media vuelve a la prevalencia, el Brier baja y el umbral teórico 0.125 vuelve a funcionar. Con 34 mil filas de entrenamiento la isotónica y Platt quedan casi empatadas; con pocos datos, Platt es más estable (animación A2.4).

## Autoverificación

In [ ]:
assert np.allclose(lr_sk.coef_[0], w_p, atol=0.02)
assert p_b.mean() > p_n.mean() + 0.1, "El modelo balanceado debería inflar la probabilidad media."
assert abs(tabla.loc["logística", "ROC-AUC"] - tabla.loc["logística balanceada", "ROC-AUC"]) < 0.02, "Ambos modelos deberían ordenar parecido."
assert tabla_cal.loc["balanceada + Platt", "Brier"] < tabla_cal.loc["balanceada sin calibrar", "Brier"], "Calibrar debería mejorar el Brier."
assert "duration" not in Xb.columns, "duration es fuga: debe excluirse."
print("✓ E2.2 completo")

## Lista de cotejo (autoevaluación)

- [ ] Tu sigmoide, log-loss y gradiente coinciden con scikit-learn.
- [ ] Explicaste el efecto de `C` y de `class_weight` sobre las probabilidades.
- [ ] Excluiste `duration` y justificaste por qué.
- [ ] Comparaste PR-AUC, Brier y el diagrama de fiabilidad en Bank Marketing.
- [ ] Decidiste por valor esperado y calibraste con Platt e isotónica.

**Reflexión:** ¿en qué decisiones de tu organización se usan probabilidades que nadie verificó si están calibradas?

**Cómo te prepara para la Tarea 2:** la tarea pide tratar el desbalance (`class_weight` y umbral) y calibrar el modelo elegido.

## Desafío opcional con IA agéntica · ¿A quién llamar? Calibración por grupo

**Objetivo.** Revisar si la calibración y la decisión por valor esperado son parecidas en distintos grupos de edad.

**Prompt inicial.** Úsalo en la herramienta que prefieras (Claude Code, Codex, Gemini en Colab, ChatGPT…), con este notebook resuelto como contexto. Pide primero un plan y revisa cada paso antes de aprobarlo.

```text
En el notebook resuelto E2.2 (regresión logística y calibración con Bank Marketing, sin la variable duration), agrega una sección que calcule el diagrama de fiabilidad y el Brier score por grupos de edad (menos de 30, de 30 a 60 y más de 60), antes y después de calibrar. Luego aplica por grupo la regla de valor esperado del notebook y explica si la decisión de a quién llamar cambia entre grupos y por qué eso importa.
```

**Cómo verificar el resultado**

- La calibración se ajusta sin tocar la prueba.
- Cada grupo se reporta con su tamaño.
- La conclusión menciona el riesgo de tratar distinto a los grupos.

**Declara el uso de IA** (norma f del sílabo): herramienta, prompts relevantes, qué verificaste tú y qué corregiste. El desafío es opcional y no se califica; lo que cuenta es que puedas explicar cada decisión.